# StatsBomb Data Inspection

Phase 1 — Data Inspection. Objective: determine whether the StatsBomb open data can supply the performance half of a player-season dataset (performance metrics per player per season), and what join keys it offers toward market-value data.

Scope: event, lineup, match, and 360-frame files under `data/statsbomb/data/`.

Key results:
- Player, match, and team identifiers exist and are unique (`player_id`, `match_id`, `team_id`).
- No market-value, transfer, fee, or wage columns exist anywhere in the dataset (proven in Cell 9).
- No biographical fields exist (no age, height, weight, or preferred foot).


## Cell 1 — Setup and data paths

Defines `DATA_DIR`, the single path variable every subsequent cell depends on, and verifies the expected directory layout (competitions file, matches, events, lineups, 360 frames) with per-directory file counts.

In [1]:
import json, os, glob
import pandas as pd
import numpy as np

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'statsbomb', 'data'))

print('exists:', os.path.isdir(DATA_DIR))
for sub in ['competitions.json', 'matches', 'events', 'lineups', 'three-sixty']:
    p = os.path.join(DATA_DIR, sub)
    n = 1 if os.path.isfile(p) else len(os.listdir(p))
    print(f'{sub}: {n}')


exists: True
competitions.json: 1
matches: 24
events: 4235
lineups: 4235
three-sixty: 426


## Cell 2 — Competitions reference

Loads `competitions.json` (80 league-season entries). The composite key `competition_id` + `season_id` identifies a league-season and maps numeric IDs to league and season names used throughout the matches data.

In [2]:
with open(os.path.join(DATA_DIR, 'competitions.json'), encoding='utf-8') as f:
    competitions = json.load(f)

df_comp = pd.json_normalize(competitions)
print(df_comp.shape)
df_comp.head()


(80, 12)


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882


## Cell 3 — Matches table

Flattens every `matches/<competition_id>/<season_id>.json` file into one wide row per match. `match_id` is the join key to the events and lineups files; team, score, stadium, and season columns provide match context.

In [3]:
match_files = glob.glob(os.path.join(DATA_DIR, 'matches', '*', '*.json'))
rows = []
for fp in match_files:
    with open(fp, encoding='utf-8') as f:
        ms = json.load(f)
    for m in ms:
        rows.append({
            'match_id': m['match_id'],
            'match_date': m['match_date'],
            'competition_id': m['competition']['competition_id'],
            'competition_name': m['competition']['competition_name'],
            'season_id': m['season']['season_id'],
            'season_name': m['season']['season_name'],
            'home_team_id': m['home_team']['home_team_id'],
            'home_team_name': m['home_team']['home_team_name'],
            'away_team_id': m['away_team']['away_team_id'],
            'away_team_name': m['away_team']['away_team_name'],
            'home_score': m['home_score'],
            'away_score': m['away_score'],
            'match_week': m.get('match_week'),
            'stadium_id': m.get('stadium', {}).get('id'),
            'stadium_name': m.get('stadium', {}).get('name'),
        })
df_matches = pd.DataFrame(rows)
print(df_matches.shape)
df_matches.head()


(3961, 15)


,match_id,match_date,competition_id,competition_name,season_id,season_name,home_team_id,home_team_name,away_team_id,away_team_name,home_score,away_score,match_week,stadium_id,stadium_name
0,9880,2018-04-14,11,La Liga,1,2017/2018,217,Barcelona,207,Valencia,2,1,32,342.0,Spotify Camp Nou
1,9912,2018-04-29,11,La Liga,1,2017/2018,219,RC Deportivo La Coruña,217,Barcelona,2,4,35,4658.0,Estadio Abanca-Riazor
2,9924,2018-05-06,11,La Liga,1,2017/2018,217,Barcelona,220,Real Madrid,2,2,36,342.0,Spotify Camp Nou
3,9855,2018-03-18,11,La Liga,1,2017/2018,217,Barcelona,215,Athletic Club,2,0,29,342.0,Spotify Camp Nou
4,9827,2018-03-01,11,La Liga,1,2017/2018,208,Las Palmas,217,Barcelona,1,1,26,357.0,Estadio de Gran Canaria


## Cell 3b — League and season coverage

Aggregates matches per league-season (where player-season rows can actually come from) and flags any `competitions.json` entries with no local match files (listed but not downloaded).

In [4]:
# Coverage: matches actually present per league-season
coverage = (df_matches
            .groupby(['competition_name', 'season_name'])
            .agg(matches=('match_id', 'nunique'))
            .reset_index()
            .sort_values(['competition_name', 'season_name']))
print(f'league-seasons with match files: {len(coverage)} (of {len(df_comp)} in competitions.json)')
coverage


league-seasons with match files: 80 (of 80 in competitions.json)


,competition_name,season_name,matches
0,1. Bundesliga,2015/2016,34
1,1. Bundesliga,2023/2024,34
2,African Cup of Nations,2023,52
3,Champions League,1970/1971,1
4,Champions League,1971/1972,1
...,...,...,...
75,UEFA Europa League,1988/1989,3
76,UEFA Women's Euro,2022,31
77,UEFA Women's Euro,2025,31
78,Women's World Cup,2019,52


The second snippet checks the reverse direction: which of the 80 `competitions.json` entries have **no** local match files. A count of zero means the local `matches/` folder fully matches the competitions reference.

In [5]:
# League-seasons listed in competitions.json but missing local files
have = set(zip(df_matches['competition_id'], df_matches['season_id']))
df_comp['has_matches'] = df_comp.apply(
    lambda r: (r['competition_id'], r['season_id']) in have, axis=1)
print('league-seasons listed but missing files:', (~df_comp['has_matches']).sum())
df_comp.loc[~df_comp['has_matches'], ['competition_name', 'season_name', 'competition_gender']]


league-seasons listed but missing files: 0


,competition_name,season_name,competition_gender


## Cell 4 — Lineups table (player registry)

Unpacks all 4,235 `lineups/<match_id>.json` files into one row per player appearance. `player_id` here is the canonical player key: it bridges to events (performance) and, later, to Transfermarkt (market value). The `match_id` is parsed from each filename.

In [6]:
lineup_files = glob.glob(os.path.join(DATA_DIR, 'lineups', '*.json'))
row_list = []
for fp in lineup_files:
    match_id = int(os.path.basename(fp).replace('.json', ''))
    with open(fp, encoding='utf-8') as f:
        teams = json.load(f)
    for t in teams:
        for p in t['lineup']:
            row_list.append({
                'match_id': match_id,
                'team_id': t['team_id'],
                'team_name': t['team_name'],
                'player_id': p['player_id'],
                'player_name': p['player_name'],
                'player_nickname': p.get('player_nickname'),
                'jersey_number': p.get('jersey_number'),
            })
df_lineups = pd.DataFrame(row_list)
print(df_lineups.shape)
df_lineups.head()


(161958, 7)


,match_id,team_id,team_name,player_id,player_name,player_nickname,jersey_number
0,15946,217,Barcelona,3109,Malcom Filipe Silva de Oliveira,Malcom,14
1,15946,217,Barcelona,3501,Philippe Coutinho Correia,Philippe Coutinho,7
2,15946,217,Barcelona,5203,Sergio Busquets i Burgos,Sergio Busquets,5
3,15946,217,Barcelona,5211,Jordi Alba Ramos,Jordi Alba,18
4,15946,217,Barcelona,5213,Gerard Piqué Bernabéu,Gerard Piqué,3


## Cell 5 — Event scale and vocabulary

Establishes the scale of the events data (~4k events per match) and the event-type vocabulary (`Pass`, `Shot`, `Carry`, `Pressure`, `Duel`, …) across a file sample, before committing to a flattened schema. Only one match is loaded at this stage.

In [7]:
event_files = glob.glob(os.path.join(DATA_DIR, 'events', '*.json'))
print('total event files:', len(event_files))

# Start small: one match
fp = event_files[0]
match_id = int(os.path.basename(fp).replace('.json', ''))
with open(fp, encoding='utf-8') as f:
    ev = json.load(f)
print('match', match_id, '->', len(ev), 'events')

# Which event types exist across a sample of files?
from collections import Counter
types_counter = Counter()
for fp in event_files[:10]:
    with open(fp, encoding='utf-8') as f:
        for e in json.load(f):
            types_counter[e['type']['name']] += 1
print(types_counter)


total event files: 4235
match 15946 -> 3762 events
Counter({'Pass': 10936, 'Ball Receipt*': 10135, 'Carry': 8989, 'Pressure': 2694, 'Ball Recovery': 861, 'Duel': 561, 'Block': 337, 'Clearance': 328, 'Dribble': 316, 'Goal Keeper': 306, 'Shot': 274, 'Foul Committed': 270, 'Foul Won': 258, 'Miscontrol': 241, 'Interception': 235, 'Dispossessed': 202, 'Dribbled Past': 181, 'Substitution': 59, 'Half Start': 40, 'Half End': 40, 'Tactical Shift': 22, 'Starting XI': 20, 'Injury Stoppage': 12, 'Shield': 12, 'Bad Behaviour': 11, '50/50': 10, 'Error': 5, 'Player Off': 3, 'Player On': 3, 'Own Goal For': 2, 'Own Goal Against': 2, 'Referee Ball-Drop': 2, 'Offside': 1})


## Cell 6 — Flattened events (long format)

Converts each nested event dict into one flat row: shared columns (`player_id`, `team_id`, `location_x/y`, …) plus type-specific payloads (`pass_length`, `shot_xg`, …). Nulls in type-specific columns are structural (a shot has no pass length), not missing data. `event_uuid` joins to the 360-frame files. New fields should be added once feature requirements are known, rather than exploding all 4,235 files upfront.

In [8]:
def flatten_event(e):
    player = e.get('player', {}) or {}
    team = e.get('team', {}) or {}
    position = e.get('position', {}) or {}
    loc = e.get('location') or [None, None]
    row = {
        'event_uuid': e.get('id'),
        'event_index': e.get('index'),
        'match_id': match_id,
        'minute': e.get('minute'),
        'second': e.get('second'),
        'period': e.get('period'),
        'type': e['type']['name'],
        'player_id': player.get('id'),
        'player_name': player.get('name'),
        'team_id': team.get('id'),
        'team_name': team.get('name'),
        'position_id': position.get('id'),
        'position': position.get('name'),
        'location_x': loc[0],
        'location_y': loc[1],
        'duration': e.get('duration'),
    }
    # Type-specific payloads (first-level only)
    if 'pass' in e:
        row['pass_length'] = e['pass'].get('length')
        row['pass_angle'] = e['pass'].get('angle')
        row['pass_height'] = e['pass'].get('height', {}).get('name')
        row['pass_recipient_id'] = e['pass'].get('recipient', {}).get('id')
    if 'shot' in e:
        row['shot_xg'] = e['shot'].get('statsbomb_xg')
        row['shot_outcome'] = e['shot'].get('outcome', {}).get('name')
        row['shot_body_part'] = e['shot'].get('body_part', {}).get('name')
        row['shot_end_x'] = e['shot'].get('end_location', [None, None])[0]
        row['shot_end_y'] = e['shot'].get('end_location', [None, None])[1]
    return row

flat = [flatten_event(e) for e in ev]
df_events = pd.DataFrame(flat)
print(df_events.shape)
df_events.head()


(3762, 25)


,event_uuid,event_index,match_id,minute,second,period,type,player_id,player_name,team_id,...,duration,pass_length,pass_angle,pass_height,pass_recipient_id,shot_xg,shot_outcome,shot_body_part,shot_end_x,shot_end_y
0,9f6e2ecf-6685-45df-a62e-c2db3090f6c1,1,15946,0,0,1,Starting XI,NaN,NaN,217,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0300039d-150d-41e4-b29a-76602ef002e6,2,15946,0,0,1,Starting XI,NaN,NaN,206,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,491e8901-7630-4cc8-b57b-937dddff2eaa,3,15946,0,0,1,Half Start,NaN,NaN,217,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,757b85ad-ddfe-44d5-b893-c23a9fb709d8,4,15946,0,0,1,Half Start,NaN,NaN,206,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,549567bd-36de-4ac8-b8dc-6b5d3f1e4be8,5,15946,0,0,1,Pass,6581.0,Jonathan Rodríguez Menéndez,206,...,2.015669,29.76995,-2.723027,Ground Pass,6855.0,NaN,NaN,NaN,NaN,NaN


## Cell 7 — Missing values and duplicates

Row counts, duplicate-key counts, and per-column null counts for all three tables. Expected findings: null `player_id` on team-level events (`Starting XI`, `Half End`) and null type-specific columns across event types are both structural, not data defects.

In [9]:
for name, df in [('matches', df_matches), ('lineups', df_lineups), ('events', df_events)]:
    print(f'--- {name} ---')
    print('rows:', len(df))
    print('dup match/event UUIDs:', df.duplicated(subset=['match_id' if name=='matches' else 'event_uuid' if name=='events' else 'player_id']).sum())
    miss = df.isna().sum()
    print('cols with missing:', miss[miss > 0].to_dict())


--- matches ---
rows: 3961
dup match/event UUIDs: 0
cols with missing: {'stadium_id': 10, 'stadium_name': 10}
--- lineups ---
rows: 161958
dup match/event UUIDs: 150069
cols with missing: {'player_nickname': 103570}
--- events ---
rows: 3762
dup match/event UUIDs: 0
cols with missing: {'player_id': 14, 'player_name': 14, 'position_id': 14, 'position': 14, 'location_x': 21, 'location_y': 21, 'duration': 1058, 'pass_length': 2599, 'pass_angle': 2599, 'pass_height': 2599, 'pass_recipient_id': 2704, 'shot_xg': 3734, 'shot_outcome': 3734, 'shot_body_part': 3734, 'shot_end_x': 3734, 'shot_end_y': 3734}


## Cell 8 — Join-key verification

Unique-value counts for `player_id`, `match_id`, `team_id`, and `event_uuid`, plus duplicate checks on `(match_id, player_id)` and `match_id`. `player_id` identifies a player across matches (not a player-season); season granularity comes from joining through matches.

In [10]:
print('lineups unique players:', df_lineups['player_id'].nunique())
print('lineups unique matches:', df_lineups['match_id'].nunique())
print('matches unique match_id:', df_matches['match_id'].nunique())
print('matches unique teams (h+a):', pd.concat([df_matches['home_team_id'], df_matches['away_team_id']]).nunique())
print('events unique event_uuid:', df_events['event_uuid'].nunique())
print('events unique player_id:', df_events['player_id'].nunique())
print()
print('dup player_id in lineups (same player twice in one match?):',
      df_lineups.duplicated(subset=['match_id', 'player_id']).sum())
print('dup match_id in matches:', df_matches['match_id'].duplicated().sum())


lineups unique players: 11889
lineups unique matches: 4235
matches unique match_id: 3961
matches unique teams (h+a): 354
events unique event_uuid: 3762
events unique player_id: 28

dup player_id in lineups (same player twice in one match?): 0
dup match_id in matches: 0


## Cell 9 — Market-value column check

Collects every top-level event key and filters for economic terms (market, transfer, value, price, fee, wage, salary). An empty result confirms StatsBomb tracks on-pitch events only: market value must be sourced externally.

In [11]:
all_keys = set()
for e in ev:
    all_keys.update(e.keys())
money_keys = [k for k in all_keys if any(s in k.lower() for s in
    ['market', 'transfer', 'value', 'price', 'fee', 'wage', 'salary'])]
print('market-value-related keys found in events:', money_keys or 'NONE — confirmed, StatsBomb has no market value data')


market-value-related keys found in events: NONE — confirmed, StatsBomb has no market value data


## Next steps

1. Inspect the Transfermarkt data the same way (`02_transfermarkt_inspection.ipynb`).
2. Establish how Transfermarkt player records join to StatsBomb `player_id`/`player_name`.
3. Build the player-season dataset: lineups (appearances) + events (performance) + Transfermarkt (value).